In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report,recall_score

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier,VotingClassifier

# Creating a Baseline Model

In [2]:
df = pd.read_csv("novagen_dataset.csv")

#df.head()
#df.isnull().sum()

#Split features
X = df.drop("Target",axis=1)
y = df["Target"]

# train test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42,stratify=y
)

In [3]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [15]:
# Logistic Regression
log_reg = LogisticRegression(
    max_iter=1000,
    penalty="l2",
    solver="liblinear"
)
log_reg.fit(X_test_scaled,y_test)

y_pred_reg = log_reg.predict(X_test_scaled)

# In Model Evaluation, Recall is more important than accuracy 
# because missing a high-risk patient is dangerous

print("Logistic reg Accuracy: ",accuracy_score(y_test,y_pred_reg))
print("Logistic reg Recall: ",recall_score(y_test,y_pred_reg))
print("Logistic reg Classification Report: ",classification_report(y_test,y_pred_reg))

Logistic reg Accuracy:  0.8267015706806282
Logistic reg Recall:  0.8483935742971888
Logistic reg Classification Report:                precision    recall  f1-score   support

           0       0.83      0.80      0.82       914
           1       0.82      0.85      0.84       996

    accuracy                           0.83      1910
   macro avg       0.83      0.83      0.83      1910
weighted avg       0.83      0.83      0.83      1910



# Model 2 kNN

In [12]:
knn = KNeighborsClassifier(
    n_neighbors=5,
    metric="euclidean"
)

knn.fit(X_train_scaled,y_train)
y_pred_knn = knn.predict(X_test_scaled)

print("KNN Accuracy: ",accuracy_score(y_test,y_pred_knn))
print("KNN Recall: ",recall_score(y_test,y_pred_knn))
print("KNN Classification Report: ",classification_report(y_test,y_pred_knn))

KNN Accuracy:  0.8832460732984293
KNN Recall:  0.8835341365461847
KNN Classification Report:                precision    recall  f1-score   support

           0       0.87      0.88      0.88       914
           1       0.89      0.88      0.89       996

    accuracy                           0.88      1910
   macro avg       0.88      0.88      0.88      1910
weighted avg       0.88      0.88      0.88      1910



# Random Forest Classifier

In [16]:
rfc = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features='sqrt',
    bootstrap=True, 
    oob_score=False
    
)

rfc.fit(X_train,y_train)

y_pred_rfc = rfc.predict(X_test)

print("Random forest Accuracy: ",accuracy_score(y_test,y_pred_rfc))
print("Random forest Recall: ",recall_score(y_test,y_pred_rfc))
print("Random forest Classification Report: ",classification_report(y_test,y_pred_rfc))



Random forest Accuracy:  0.9345549738219895
Random forest Recall:  0.9528112449799196
Random forest Classification Report:                precision    recall  f1-score   support

           0       0.95      0.91      0.93       914
           1       0.92      0.95      0.94       996

    accuracy                           0.93      1910
   macro avg       0.94      0.93      0.93      1910
weighted avg       0.93      0.93      0.93      1910



# Gradient Boosting (Ensemble Learning)

In [23]:
gb = GradientBoostingClassifier(
    learning_rate=0.1, 
    n_estimators=150,
    random_state=42,
    max_depth=3
)

gb.fit(X_train,y_train)

y_pred_gb = gb.predict(X_test)

print("Gradient boosting Accuracy: ",accuracy_score(y_test,y_pred_gb))
print("Gradient boosting Recall: ",recall_score(y_test,y_pred_gb))
print("Gradient boosting Classification Report: ",classification_report(y_test,y_pred_gb))


Gradient boosting Accuracy:  0.9303664921465968
Gradient boosting Recall:  0.9497991967871486
Gradient boosting Classification Report:                precision    recall  f1-score   support

           0       0.94      0.91      0.93       914
           1       0.92      0.95      0.93       996

    accuracy                           0.93      1910
   macro avg       0.93      0.93      0.93      1910
weighted avg       0.93      0.93      0.93      1910



# Voting classifier

In [22]:
vc = VotingClassifier(
    estimators=[
        ("lr",log_reg),
        ("knn",knn),
        ("gb",gb)
    ],
    voting="soft"
)

vc.fit(X_train_scaled, y_train)

y_pred_vote = vc.predict(X_test_scaled)

print("Voting Classifier Accuracy:", accuracy_score(y_test, y_pred_vote))
print("Voting Classifier Recall:", recall_score(y_test, y_pred_vote))
print(classification_report(y_test, y_pred_vote))

Voting Classifier Accuracy: 0.9162303664921466
Voting Classifier Recall: 0.929718875502008
              precision    recall  f1-score   support

           0       0.92      0.90      0.91       914
           1       0.91      0.93      0.92       996

    accuracy                           0.92      1910
   macro avg       0.92      0.92      0.92      1910
weighted avg       0.92      0.92      0.92      1910



# Results


| Model                | Recall |
|----------------------|:------:|
| Logistic Regression  | 84.8%  |
| KNN                  | 88.3%  |
| Random Forest        | 95.2%  |
| Gradient Boosting    | 94.9%  |
| Voting Classifier    | 92.9% |




### Best Classifier that we should use for NovaGen(based on Recall) - Random Forest with accuracy of 95.2%